In [1]:
import torch, torchvision, time, os, json, copy, random
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
from thop import profile
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

g = torch.Generator(); g.manual_seed(SEED)

# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
BASELINE    = "../__2__baseline_resnet50_cifar10.pth"
OUTPUT_JSON = "__structured_pruning_results.json"
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

CHANNEL_PRUNE_RATIO = 0.30
FINETUNE_EPOCHS     = 10
FINETUNE_LR         = 1e-3
TIMING_WARMUP       = 50
TIMING_REPS         = 500

print(f"Device           : {DEVICE}")
print(f"Channel ratio    : {CHANNEL_PRUNE_RATIO*100:.0f}%")
print(f"Fine-tune budget : {FINETUNE_EPOCHS} epochs @ lr={FINETUNE_LR}")

# ── MODEL FACTORY ─────────────────────────────────────────────
def build_model(num_classes=NUM_CLASSES):
    m = models.resnet50(pretrained=False)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m

def load_baseline():
    m = build_model()
    m.load_state_dict(torch.load(BASELINE, map_location=DEVICE))
    return m.to(DEVICE)

# ── DATA ──────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

train_set    = torchvision.datasets.CIFAR10('../data', True,  download=True,
                                            transform=transform_train)
test_set     = torchvision.datasets.CIFAR10('../data', False, download=True,
                                            transform=transform_test)
train_loader = torch.utils.data.DataLoader(
    train_set, BATCH_SIZE, shuffle=True,  num_workers=0,
    worker_init_fn=seed_worker, generator=g)
test_loader  = torch.utils.data.DataLoader(
    test_set,  BATCH_SIZE, shuffle=False, num_workers=0)

# ── METRIC HELPERS ────────────────────────────────────────────
def evaluate_full(model):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            preds.extend(model(x.to(DEVICE)).argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    p, l = np.array(preds), np.array(labels)
    return dict(
        accuracy  = float(accuracy_score(l, p)),
        precision = float(precision_score(l, p, average='macro', zero_division=0)),
        recall    = float(recall_score(l, p, average='macro', zero_division=0)),
        f1        = float(f1_score(l, p, average='macro', zero_division=0)),
    )

def count_params(model):  return int(sum(p.numel() for p in model.parameters()))
def count_nonzero(model): return int(sum((p != 0).sum().item() for p in model.parameters()))

def model_size_mb(model, path="_tmp_.pth"):
    torch.save(model.state_dict(), path)
    mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return float(mb)

def compute_flops(model):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return float(macs * 2 / 1e9)

def inference_ms_fn(model):
    """Single-sample latency with CUDA synchronization."""
    model.eval()
    dummy  = torch.randn(1, 3, 32, 32).to(DEVICE)
    is_gpu = DEVICE.type == "cuda"
    with torch.no_grad():
        for _ in range(TIMING_WARMUP):
            model(dummy)
        if is_gpu: torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(TIMING_REPS):
            model(dummy)
        if is_gpu: torch.cuda.synchronize()
    return float((time.perf_counter() - t0) / TIMING_REPS * 1000)

def collect_metrics(model):
    """All metrics including inference_ms and theoretical size."""
    m   = evaluate_full(model)
    tot = count_params(model)
    nz  = count_nonzero(model)
    # BUG FIX 4: compute theoretical_size_mb (nonzero float32 weights only)
    theoretical_size_mb = (nz * 4) / 1e6
    m.update(dict(
        params_total          = tot,
        params_nonzero        = nz,
        sparsity_actual       = float(1 - nz / tot),
        size_mb               = model_size_mb(model),
        size_mb_theoretical   = round(theoretical_size_mb, 4),
        flops_G               = compute_flops(model),
        inference_ms          = inference_ms_fn(model),   # BUG FIX 2: always populated
    ))
    return m

# ── TRAINING UTILITIES ────────────────────────────────────────
def train_model(model, epochs, lr=0.1, desc="train"):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
        scheduler.step()
        if (ep + 1) % 5 == 0 or ep == 0:
            acc = evaluate_full(model)["accuracy"]
            print(f"    [{desc}] ep {ep+1:>3}/{epochs}  acc={acc:.4f}")
    return model

def fine_tune(model, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR, desc="ft"):
    return train_model(model, epochs, lr=lr, desc=desc)

# ── CHECKPOINT SAVING ─────────────────────────────────────────
SAVE_DIR = "saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_model(model, key):                          # BUG FIX 1: takes model explicitly
    path = os.path.join(SAVE_DIR, f"{key}.pth")
    torch.save(model.state_dict(), path)
    print(f"  ✓ Saved → {path}")

# ── STRUCTURED PRUNING HELPERS ────────────────────────────────
def _prune_conv_out(conv, kept_idx):
    n   = len(kept_idx)
    new = nn.Conv2d(conv.in_channels, n, conv.kernel_size, conv.stride,
                    conv.padding, groups=conv.groups,
                    bias=(conv.bias is not None))
    new.weight.data = conv.weight.data[kept_idx].clone()
    if conv.bias is not None:
        new.bias.data = conv.bias.data[kept_idx].clone()
    return new

def _prune_conv_in(conv, kept_idx):
    new = nn.Conv2d(len(kept_idx), conv.out_channels, conv.kernel_size,
                    conv.stride, conv.padding, groups=conv.groups,
                    bias=(conv.bias is not None))
    new.weight.data = conv.weight.data[:, kept_idx].clone()
    if conv.bias is not None:
        new.bias.data = conv.bias.data.clone()
    return new

def _prune_bn(bn, kept_idx):
    n   = len(kept_idx)
    new = nn.BatchNorm2d(n, eps=bn.eps, momentum=bn.momentum,
                          affine=bn.affine,
                          track_running_stats=bn.track_running_stats)
    if bn.affine:
        new.weight.data = bn.weight.data[kept_idx].clone()
        new.bias.data   = bn.bias.data[kept_idx].clone()
    if bn.track_running_stats:
        new.running_mean        = bn.running_mean[kept_idx].clone()
        new.running_var         = bn.running_var[kept_idx].clone()
        new.num_batches_tracked = bn.num_batches_tracked.clone()
    return new

def prune_resnet50_structured(model, ratio):
    """
    Prune internal Bottleneck width (conv1→conv2) by L2-norm channel scoring.
    conv3, bn3, and downsample are NEVER touched → residual shapes valid.
    Shape consistency asserted per block.
    """
    model = copy.deepcopy(model)
    for stage in ['layer1', 'layer2', 'layer3', 'layer4']:
        for block in getattr(model, stage):
            conv1  = block.conv1
            bn1    = block.bn1
            conv2  = block.conv2
            scores = conv1.weight.data.view(conv1.out_channels, -1).norm(p=2, dim=1)
            n_keep = max(1, int(conv1.out_channels * (1 - ratio)))
            kept   = scores.argsort(descending=True)[:n_keep].sort().values
            block.conv1 = _prune_conv_out(conv1, kept)
            block.bn1   = _prune_bn(bn1, kept)
            block.conv2 = _prune_conv_in(conv2, kept)
            assert block.conv1.out_channels == block.conv2.in_channels, (
                f"Shape mismatch after pruning {stage}: "
                f"conv1.out={block.conv1.out_channels} "
                f"conv2.in={block.conv2.in_channels}"
            )
    return model

# ══════════════════════════════════════════════════════════════
# METHOD 2 — STRUCTURED PRUNING
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("2. STRUCTURED PRUNING  (channel removal + fine-tune)")
print("="*65)

model_struct = prune_resnet50_structured(
    load_baseline(), ratio=CHANNEL_PRUNE_RATIO).to(DEVICE)

model_struct = fine_tune(model_struct, epochs=FINETUNE_EPOCHS,
                          lr=FINETUNE_LR, desc="2-structured-ft")

results = {}
results["2_structured"] = collect_metrics(model_struct)
results["2_structured"]["flops_note"] = (
    f"TRUE structured pruning: {CHANNEL_PRUNE_RATIO*100:.0f}% of internal "
    "Bottleneck channels physically removed (conv1+bn1+conv2 per block). "
    "conv3, bn3, downsample untouched — residual shapes guaranteed. "
    "GFLOPs genuinely reduced; measured by thop on rebuilt model."
)

save_model(model_struct, "2_structured")   # BUG FIX 1: was save_model(model, ...)

r = results["2_structured"]
print(f"\n  acc       = {r['accuracy']:.4f}")
print(f"  f1        = {r['f1']:.4f}")
print(f"  sparsity  = {r['sparsity_actual']:.3f}")
print(f"  size_mb   = {r['size_mb']:.2f} MB")
print(f"  size_theo = {r['size_mb_theoretical']:.2f} MB  (nonzero float32 only)")
print(f"  flops     = {r['flops_G']:.3f} GFLOPs")
print(f"  infer_ms  = {r['inference_ms']:.3f} ms")

# ── SAVE JSON ─────────────────────────────────────────────────
output = {
    "_meta": {
        "framework"           : "Structured Pruning Only",
        "baseline_pth"        : BASELINE,
        "channel_prune_ratio" : CHANNEL_PRUNE_RATIO,
        "device"              : str(DEVICE),
        "seed"                : SEED,
        "finetune_epochs"     : FINETUNE_EPOCHS,
        "finetune_lr"         : FINETUNE_LR,
        "timing_config": {
            "warmup_reps": TIMING_WARMUP,
            "timed_reps" : TIMING_REPS,
            "cuda_sync"  : True,
        },
        "structured_pruning_safety": (
            "Only internal Bottleneck width (conv1→conv2) is pruned. "
            "conv3, bn3, downsample untouched — residual shape safety guaranteed."
        ),
    },
    "results": results
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output, f, indent=2)
print(f"\n✓ Saved → {OUTPUT_JSON}")

# ── VERIFICATION ──────────────────────────────────────────────
# BUG FIX 5: only check things that exist for structured pruning
print("\n" + "="*65)
print("VERIFICATION")
print("="*65)
ok = True

# V1: GFLOPs reduced vs baseline dense model (proxy: structured < 4.1 GFLOPs)
DENSE_FLOPS_APPROX = 4.1   # ResNet-50 CIFAR-10 dense baseline
v1 = r['flops_G'] < DENSE_FLOPS_APPROX
print(f"\nV1  Structured GFLOPs {r['flops_G']:.3f} < dense baseline ~{DENSE_FLOPS_APPROX}")
print(f"    → {'PASS ✓' if v1 else 'FAIL — FLOPs not reduced'}")
ok = ok and v1

# V2: Sparsity is nonzero (channels were physically removed → fewer params)
v2 = r['sparsity_actual'] > 0.0
print(f"\nV2  Sparsity > 0: {r['sparsity_actual']:.3f}")
print(f"    → {'PASS ✓' if v2 else 'FAIL — no parameters removed'}")
ok = ok and v2

# V3: size_mb_theoretical <= size_mb (theoretical is always a lower bound)
v3 = r['size_mb_theoretical'] <= r['size_mb']
print(f"\nV3  Theoretical size {r['size_mb_theoretical']:.2f} MB <= actual {r['size_mb']:.2f} MB")
print(f"    → {'PASS ✓' if v3 else 'FAIL — theoretical exceeds actual (float type mismatch?)'}")
ok = ok and v3

# V4: inference_ms is populated
v4 = r['inference_ms'] is not None and r['inference_ms'] > 0
print(f"\nV4  inference_ms populated: {r['inference_ms']:.3f} ms")
print(f"    → {'PASS ✓' if v4 else 'FAIL — inference_ms is None or zero'}")
ok = ok and v4

# V5: Shape sanity — forward pass on dummy input succeeds
try:
    model_struct.eval()
    with torch.no_grad():
        out = model_struct(torch.randn(2, 3, 32, 32).to(DEVICE))
    v5 = out.shape == (2, NUM_CLASSES)
    print(f"\nV5  Forward pass output shape: {tuple(out.shape)}  (expected (2, {NUM_CLASSES}))")
    print(f"    → {'PASS ✓' if v5 else 'FAIL — wrong output shape'}")
except Exception as e:
    v5 = False
    print(f"\nV5  Forward pass FAILED: {e}")
ok = ok and v5

print("\n" + "="*65)
print(f"OVERALL: {'ALL CHECKS PASSED ✓' if ok else 'SOME CHECKS FAILED — review above'}")
print("="*65)

Device           : cuda
Channel ratio    : 30%
Fine-tune budget : 10 epochs @ lr=0.001


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



2. STRUCTURED PRUNING  (channel removal + fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [2-structured-ft] ep   1/10  acc=0.9297
    [2-structured-ft] ep   5/10  acc=0.9319
    [2-structured-ft] ep  10/10  acc=0.9317
  ✓ Saved → saved_models\2_structured.pth

  acc       = 0.9317
  f1        = 0.9317
  sparsity  = 0.000
  size_mb   = 75.52 MB
  size_theo = 75.23 MB  (nonzero float32 only)
  flops     = 2.069 GFLOPs
  infer_ms  = 4.853 ms

✓ Saved → __structured_pruning_results.json

VERIFICATION

V1  Structured GFLOPs 2.069 < dense baseline ~4.1
    → PASS ✓

V2  Sparsity > 0: 0.000
    → FAIL — no parameters removed

V3  Theoretical size 75.23 MB <= actual 75.52 MB
    → PASS ✓

V4  inference_ms populated: 4.853 ms
    → PASS ✓

V5  Forward pass output shape: (2, 10)  (expected (2, 10))
    → PASS ✓

OVERALL: SOME CHECKS FAILED — review above
